# EP3 — Segmentación de Pacientes y Predicción de Hospitalización

Este notebook desarrolla una solución de Machine Learning aplicada a un problema de salud: **segmentar pacientes en perfiles clínicos y predecir si un paciente será hospitalizado** a partir de sus indicadores de salud.


> **Nota sobre los datos (importante):** el dataset `pacientes.csv` es una **simulación académica** generada para este trabajo. Los valores se construyeron a partir de cuatro perfiles clínicos coherentes (paciente saludable joven, riesgo metabólico, adulto controlado y mayor frágil), con **ruido controlado** para que no quede artificialmente perfecto. Además, se introdujeron de forma deliberada **valores nulos, filas duplicadas, valores atípicos (outliers) y texto inconsistente**, de modo que la fase de limpieza reproduzca el trabajo real con datos sucios. Se declara explícitamente como datos simulados, siguiendo el mismo criterio de simulación usado en la evaluación anterior. La variable objetivo es `hospitalizado` (0 = no, 1 = sí), por lo que el problema supervisado es de **clasificación**.

Integrantes:

- Nicolas Salas
- Francisco Gaete
- Nicolás Cisternas

## 1. Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
print("Librerías importadas correctamente.")

## 2. Cargar el dataset

Cargamos `pacientes.csv` (ajusta la ruta a la carpeta `/data` del repositorio).

In [ ]:
df = pd.read_csv("data\pacientes.csv")
df.head()

## 3. Inspección inicial del dataset

In [ ]:
print("Dimensión del dataset (filas, columnas):")
print(df.shape)

In [ ]:
print("Información general (tipos y no nulos):")
df.info()

In [ ]:
print("Resumen estadístico de columnas numéricas:")
df.describe()

## 4. Distribución de la variable objetivo

Revisamos el balance entre pacientes hospitalizados y no hospitalizados. Mirar la distribución es una buena práctica inicial: detecta si el problema está muy desbalanceado antes de entrenar.

In [ ]:
print("Cantidad por clase:")
print(df["hospitalizado"].value_counts())
print()
print("Proporción:")
print(df["hospitalizado"].value_counts(normalize=True).round(3))

## 5. Detección de duplicados

Revisamos si hay filas repetidas. Comparamos ignorando `id_paciente`, porque ese identificador es único por fila aunque el resto de los datos sea idéntico.

In [ ]:
cols_sin_id = [c for c in df.columns if c != "id_paciente"]
duplicados = df.duplicated(subset=cols_sin_id).sum()
print("Cantidad de filas duplicadas (ignorando id_paciente):", duplicados)

## 6. Detección de valores nulos

Revisamos cuántos valores faltantes hay por columna. Los datos reales casi siempre traen vacíos por errores de registro.

In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum())
print("\nTotal de nulos:", df.isnull().sum().sum())

## 7. Normalización de texto

Las columnas de texto vienen con inconsistencias (mayúsculas, minúsculas, espacios, sinónimos). Por ejemplo, el sexo aparece como `F`, `f`, ` F `, `femenino`. Las unificamos a un formato estándar.

In [ ]:
print("Antes de normalizar:")
print("  sexo:", df["sexo"].unique())
print("  fuma:", df["fuma"].unique())

# Limpiar espacios y unificar a minúsculas, luego mapear a un valor estándar
df["sexo"] = (df["sexo"].astype(str).str.strip().str.lower()
              .map({"f": "F", "femenino": "F", "m": "M", "masculino": "M"}))
df["fuma"] = (df["fuma"].astype(str).str.strip().str.lower()
              .map({"si": "Si", "no": "No"}))

print("\nDespués de normalizar:")
print("  sexo:", df["sexo"].unique())
print("  fuma:", df["fuma"].unique())

## 8. Corrección de edades imposibles

Algunas edades son claramente errores de registro (valores como 3, 150 o 999 años). Para un estudio de pacientes adultos, tratamos como inválida cualquier edad fuera del rango 18–110 y la marcamos como nula (se rellenará en el paso siguiente).

In [ ]:
import numpy as np
antes = ((df["edad"] < 18) | (df["edad"] > 110)).sum()
df.loc[(df["edad"] < 18) | (df["edad"] > 110), "edad"] = np.nan
print("Edades imposibles corregidas (pasadas a nulo):", int(antes))
print("Rango de edad válido ahora:", df["edad"].min(), "-", df["edad"].max())

## 9. Eliminación de columnas sin valor analítico

`id_paciente` solo identifica a la persona (como `id_cliente` en clase: no describe nada clínico), por lo que no sirve para agrupar ni predecir.

In [ ]:
df = df.drop(columns=["id_paciente"])
print("Columnas tras la eliminación:", df.shape[1])

## 10. Eliminación de duplicados

Quitamos las filas repetidas detectadas antes.

In [ ]:
filas_antes = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
print("Filas eliminadas por duplicados:", filas_antes - df.shape[0])
print("Filas restantes:", df.shape[0])

## 11. Tratamiento de valores nulos

Rellenamos los nulos en lugar de eliminar filas (perderíamos muchos datos). Para las variables **numéricas** usamos la **mediana** (robusta a outliers); para las **categóricas**, el valor más frecuente (la moda).

In [ ]:
# Numéricas -> mediana
num_cols = df.select_dtypes(include=[np.number]).columns
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

# Categóricas -> moda
cat_cols = df.select_dtypes(include="object").columns
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

print("Nulos restantes tras el relleno:", df.isnull().sum().sum())

## 12. Tratamiento de outliers

Variables clínicas como presión, glucosa o triglicéridos traen valores extremos (errores de medición). Filtramos por el percentil 99 de cada una para que no distorsionen el clustering ni los modelos.

In [ ]:
filas_antes = df.shape[0]
for col in ["presion_sistolica", "glucosa", "trigliceridos", "colesterol", "imc"]:
    limite = df[col].quantile(0.99)
    df = df[df[col] <= limite]
df = df.reset_index(drop=True)
print("Filas eliminadas por outliers:", filas_antes - df.shape[0])
print("Filas restantes:", df.shape[0])

## 13. Verificación posterior a la limpieza

In [ ]:
print("Nulos totales:", df.isnull().sum().sum())
print("Duplicados:", df.duplicated().sum())
print("Dimensiones finales:", df.shape)

## 14. Creación de variables nuevas (feature engineering)

Creamos variables categóricas legibles, útiles para los gráficos. Se crean sobre `df` (versión con texto), pero NO se usan como entrada de los modelos.

**1. Rango etario** (Joven / Adulto / Mayor) a partir de `edad`.

In [ ]:
def clasificar_edad(x):
    if x < 40:
        return "Joven"
    elif x < 65:
        return "Adulto"
    else:
        return "Mayor"

df["rango_etario"] = df["edad"].apply(clasificar_edad)
print(df["rango_etario"].value_counts())

**2. Rango de IMC** agrupado en categorías de salud estándar.

In [ ]:
def clasificar_imc(x):
    if x < 18.5:
        return "Bajo peso"
    elif x < 25:
        return "Normal"
    elif x < 30:
        return "Sobrepeso"
    else:
        return "Obesidad"

df["rango_imc"] = df["imc"].apply(clasificar_imc)
print(df["rango_imc"].value_counts())

## 15. Codificación de categóricas con get_dummies

Para los modelos todo debe ser numérico. Aplicamos one-hot encoding sobre una **copia** (`df_modelo`), dejando `df` intacto con el texto para los gráficos. No codificamos los rangos creados (son para graficar).

In [ ]:
df_modelo = df.copy()

categoricas = df_modelo.select_dtypes(include="object").columns.tolist()
print("Columnas categóricas detectadas:", categoricas)

cols_a_codificar = [c for c in categoricas if c not in ["rango_etario", "rango_imc"]]
df_modelo = pd.get_dummies(df_modelo, columns=cols_a_codificar, drop_first=True)
df_modelo = df_modelo.drop(columns=["rango_etario", "rango_imc"])

print("Dimensiones de df_modelo:", df_modelo.shape)
df_modelo.head()

## 16. Selección de variables para clustering

Buscamos grupos de pacientes parecidos **sin usar la etiqueta** `hospitalizado`. Usamos los indicadores clínicos numéricos. No incluimos `hospitalizado`: en clustering no existe `y`, buscamos grupos, no predecimos.

In [ ]:
variables_cluster = ["edad", "imc", "presion_sistolica", "presion_diastolica",
                     "glucosa", "colesterol", "trigliceridos", "hdl",
                     "horas_actividad_semana", "cigarrillos_dia"]
X_cluster = df[variables_cluster].copy()
print("Variables usadas para clustering:", len(variables_cluster))
X_cluster.head()

## 17. Escalar variables

K-Means calcula distancias. Si una variable tiene números grandes y otra pequeños, la grande domina solo por su escala. `StandardScaler` deja todas en una cancha comparable. No borra información; solo nivela la cancha.

In [ ]:
scaler = StandardScaler()
X_cluster_escalado = scaler.fit_transform(X_cluster)
print("Datos escalados. Forma:", X_cluster_escalado.shape)

## 18. Evaluación de K-Means con varios valores de K

Para cada K calculamos la **Inertia** (compacidad; baja siempre al subir K) y el **Silhouette** (qué tan bien ubicado está cada punto; de -1 a 1, más alto es mejor).

In [ ]:
rango_k = range(2, 8)
inercias, silhouettes = [], []
for k in rango_k:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    etiquetas = km.fit_predict(X_cluster_escalado)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cluster_escalado, etiquetas,
                                        sample_size=5000, random_state=RANDOM_STATE))

tabla_k = pd.DataFrame({
    "K": list(rango_k),
    "Inertia": [round(i, 1) for i in inercias],
    "Silhouette": [round(s, 3) for s in silhouettes]
})
print(tabla_k.to_string(index=False))

## 19. Gráficos para justificar K

In [ ]:
fig_codo = px.line(tabla_k, x="K", y="Inertia",
    title="Método del codo (Inertia por K)",
    labels={"K": "Cantidad de clusters (K)", "Inertia": "Inertia"}, markers=True)
fig_codo.show()

In [ ]:
fig_sil = px.line(tabla_k, x="K", y="Silhouette",
    title="Silhouette score por K",
    labels={"K": "Cantidad de clusters (K)", "Silhouette": "Silhouette score"}, markers=True)
fig_sil.show()

### Interpretación de la elección de K

Como vimos en clase, **K lo decide el analista**, justificándolo con un argumento visual (gráficos) y uno numérico (tabla). En este dataset, **K=4 obtiene el mejor Silhouette**, claramente por encima de los demás valores. Esto indica cuatro grupos de pacientes bien diferenciados.

El método del codo también muestra una mejora fuerte hasta K=4 y luego se aplana, lo que refuerza la elección. Por eso K=4 combina el mejor argumento numérico y visual.

## 20. Aplicación final de K-Means con el K elegido

In [ ]:
K_FINAL = int(tabla_k.loc[tabla_k["Silhouette"].idxmax(), "K"])
print("K elegido (mejor Silhouette):", K_FINAL)

kmeans_final = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = kmeans_final.fit_predict(X_cluster_escalado)

print("\nCantidad de pacientes por cluster:")
print(df["cluster"].value_counts().sort_index())

sil_final = silhouette_score(X_cluster_escalado, df["cluster"],
                             sample_size=5000, random_state=RANDOM_STATE)
print("\nSilhouette del modelo final:", round(sil_final, 3))